### csv and Excel files manipulation


In [2]:
import pandas as pd
import os


In [3]:
os.makedirs("data/structured_files",exist_ok =True)

In [4]:
data ={
    'product_name': ['laptop','mouse','keyboard','monitor','webcame'],
    'category': ['electronics','accessories','accessories','electronics','electronics'],
    'price': [1000, 50, 30, 300, 150],
    'quantity': [10, 200, 150, 50, 100]
}
# save csv files
df=pd.DataFrame(data)
df.to_csv("data/structured_files/products.csv",index=False)

In [7]:
with pd.ExcelWriter("data/structured_files/products.xlsx") as writer:
    df.to_excel(writer, sheet_name='Products', index=False)
    
    summary_data={
        'total_products': df.shape[0],
        'total_categories': df['category'].nunique(),
        'total_revenue': (df['price'] * df['quantity']).sum()   
    }
    pd.DataFrame([summary_data]).to_excel(writer, sheet_name='Summary', index=False)

In [8]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader

In [13]:
#method 1: CSVLoader - each row becomes a documents
print("1 csv loader - Row-based-Documents")
csv_loader = CSVLoader(file_path="data/structured_files/products.csv", encoding='utf-8', csv_args={"delimiter": ",", "quotechar": '"' })

csv_docs=csv_loader.load()
print(f"loaded {len(csv_docs)} documents (one per row)")
print(f"content : {csv_docs[0].page_content }")
print(f"Metadata : {csv_docs[0].metadata}")

1 csv loader - Row-based-Documents
loaded 5 documents (one per row)
content : product_name: laptop
category: electronics
price: 1000
quantity: 10
Metadata : {'source': 'data/structured_files/products.csv', 'row': 0}


In [16]:
print(csv_docs)

[Document(metadata={'source': 'data/structured_files/products.csv', 'row': 0}, page_content='product_name: laptop\ncategory: electronics\nprice: 1000\nquantity: 10'), Document(metadata={'source': 'data/structured_files/products.csv', 'row': 1}, page_content='product_name: mouse\ncategory: accessories\nprice: 50\nquantity: 200'), Document(metadata={'source': 'data/structured_files/products.csv', 'row': 2}, page_content='product_name: keyboard\ncategory: accessories\nprice: 30\nquantity: 150'), Document(metadata={'source': 'data/structured_files/products.csv', 'row': 3}, page_content='product_name: monitor\ncategory: electronics\nprice: 300\nquantity: 50'), Document(metadata={'source': 'data/structured_files/products.csv', 'row': 4}, page_content='product_name: webcame\ncategory: electronics\nprice: 150\nquantity: 100')]


In [18]:
from typing import List
from langchain_core.documents import Document


### Method 2 Unstructured CsvLoader

In [26]:
def process_csv_intelligently(filepath: str) -> List[Document]:
    """
    Process CSV file intelligently
    """
    df = pd.read_csv(filepath)

    documents = []

    for idx, row in df.iterrows():
        content = f"Product Name: {row['product_name']}, Category: {row['category']}, Price: {row['price']}, Quantity: {row['quantity']}"
        metadata = {
            "product_name": row['product_name'],
            "category": row['category'],
            "price": row['price'],
            "quantity": row['quantity']
        }
        documents.append(Document(page_content=content, metadata=metadata))

        doc = Document(page_content=content, metadata={'source':filepath,
                                                       'row_index': idx,
                                                       'product_name': row['product_name'],
                                                       'category': row['category'],
                                                       'price': row['price'],
                                                       'quantity': row['quantity']})
        documents.append(doc)
    return documents

In [27]:
process_csv_intelligently("data/structured_files/products.csv")

[Document(metadata={'product_name': 'laptop', 'category': 'electronics', 'price': 1000, 'quantity': 10}, page_content='Product Name: laptop, Category: electronics, Price: 1000, Quantity: 10'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 0, 'product_name': 'laptop', 'category': 'electronics', 'price': 1000, 'quantity': 10}, page_content='Product Name: laptop, Category: electronics, Price: 1000, Quantity: 10'),
 Document(metadata={'product_name': 'mouse', 'category': 'accessories', 'price': 50, 'quantity': 200}, page_content='Product Name: mouse, Category: accessories, Price: 50, Quantity: 200'),
 Document(metadata={'source': 'data/structured_files/products.csv', 'row_index': 1, 'product_name': 'mouse', 'category': 'accessories', 'price': 50, 'quantity': 200}, page_content='Product Name: mouse, Category: accessories, Price: 50, Quantity: 200'),
 Document(metadata={'product_name': 'keyboard', 'category': 'accessories', 'price': 30, 'quantity': 150}, pa

csv processing Strategies

1.Row based(CASVLOader):
 simple one-row-one-document
 good for record lookups
 loses table context

2.context Processing
    preserves relationships
    create summaries
    Rich metadata 
    Better for Q&A